# Synthetic Global Collectible RPG Analytics

This notebook analyzes a fictional character-collection, turn-based mobile PvE RPG in KR, JP, and Global West. All data and incidents are synthetic.

## 1. Setup and deterministic generation

In [ ]:
import sys
import pandas as pd
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.generate_synthetic_data import main as generate_synthetic_data
from src.analyze_game_data import (
    event_window_summary, load_data, prepare_metrics,
    product_summary, save_charts, validate_data, validate_derived_data,
)
from src.analyze_lifecycle import (
    event_dependency_summary, lifecycle_event_performance,
    lifecycle_phase_summary, save_lifecycle_charts,
)
from src.analyze_retention import (
    acquisition_quality_comparison, retention_monthly_summary,
    regional_quality_summary, save_retention_charts,
)
from src.analyze_pve import (
    boss_performance_summary, boss_regional_summary, event_pve_alignment,
    prepare_boss_funnel, save_pve_charts,
)
from src.analyze_monetization import (
    adjacent_product_summary, bm_evaluation_summary,
    monetization_window_summary, prepare_sales, revenue_decomposition,
    save_monetization_charts,
)
from src.analyze_incident import (
    incident_final_evaluation, incident_retention_summary,
    incident_stage_evaluation, incident_window_summary,
    save_incident_charts,
)
from src.analyze_regional import (
    regional_action_plan, regional_evidence_summary,
    regional_guardrail_matrix, save_regional_charts,
)

generate_synthetic_data()

## 2. Quality validation and KPI preparation

In [ ]:
daily, retention, events, products, sales, bosses = load_data(PROJECT_ROOT)
quality_report = validate_data(daily, retention, events, products, sales, bosses)
daily_metrics, retention_metrics, boss_metrics = prepare_metrics(daily, retention, bosses)
quality_report

## 3. Pre-registered decision framework

`H0-S` states that no observable KPI difference exists from the stated baseline. `H0-B` states that any difference is too small, too short-lived, or offset by a guardrail failure to matter operationally. Routine events use the preceding 14 days; incident recovery uses the clean July 31–August 11 pre-incident baseline. Practical thresholds and baseline variability drive descriptive scenario evaluation; these comparisons are not causal estimates.

In [ ]:
event_results = event_window_summary(daily_metrics, events)
event_results[event_results['event_name'].isin([
    'PvE Growth Subscription Launch', 'Astra Heroes Crossover',
    'Data Center Outage', 'Extraordinary Compensation',
    'Postmortem and Trust Recovery',
])][[
    'event_name', 'dau_change_pct', 'returned_users_change_pct',
    'revenue_change_pct', 'conversion_rate_change_pct',
]].round(2)

derived_quality_report = validate_derived_data(
    daily_metrics, retention_metrics, event_results, boss_metrics,
)
derived_quality_report

In [ ]:
product_results = product_summary(sales, products)
product_results

In [ ]:
save_charts(
    daily_metrics, retention_metrics, events, event_results,
    product_results, boss_metrics, PROJECT_ROOT / 'images',
)
print('Charts regenerated in:', PROJECT_ROOT / 'images')

## 4. Analysis 1 — lifecycle and live-ops dependence

This analysis separates immediate lift from post-event durability, reports baseline and post-window overlaps, and measures KPI concentration on planned live-ops days.

In [ ]:
lifecycle_events = lifecycle_event_performance(daily_metrics, events)
lifecycle_events[[
    'event_name', 'dau_during_change_pct', 'dau_post_14_change_pct',
    'baseline_contaminated', 'post_14_contaminated',
    'immediate_result', 'durability_result',
]].round(2)

In [ ]:
phase_results = lifecycle_phase_summary(daily_metrics)
dependency_results = event_dependency_summary(daily_metrics)
dependency_results[dependency_results['scope'].eq('ALL')].round(3)

In [ ]:
save_lifecycle_charts(
    daily_metrics, lifecycle_events, dependency_results, PROJECT_ROOT / 'images',
)
print('Analysis 1 charts regenerated in:', PROJECT_ROOT / 'images')

## 5. Analysis 2 — acquisition quality and retention

Monthly cohorts are treated as event context rather than event-level causal attribution. Count-weighted retention and the D30 retained-user gap at target volume separate acquisition scale from long-term quality without implying causality.

In [ ]:
retention_monthly = retention_monthly_summary(retention, events)
acquisition_quality = acquisition_quality_comparison(retention, retention_monthly)
acquisition_quality[[
    'comparison_label', 'cohort_size_change_pct',
    'd1_retention_change_pp', 'd7_retention_change_pp',
    'd30_retention_change_pp',
    'd30_retained_gap_vs_baseline_quality_at_target_volume', 'outcome',
]].round(2)

In [ ]:
save_retention_charts(
    retention_monthly, acquisition_quality, PROJECT_ROOT / 'images',
)
print('Analysis 2 charts regenerated in:', PROJECT_ROOT / 'images')

## 6. Analysis 3 — core-content entry and PvE engagement

Difficulty-level participant counts may overlap and are never summed into unique users. NORMAL participation is the broadest entry proxy; clear rate and attempt burden describe outcomes after entry. The first three bosses form the reference benchmark, and Astra is the held-out comparison.

In [ ]:
boss_funnel = prepare_boss_funnel(bosses, daily)
boss_summary = boss_performance_summary(boss_funnel)
boss_regional = boss_regional_summary(boss_funnel)
pve_alignment = event_pve_alignment(
    boss_summary, lifecycle_events, acquisition_quality,
)
pve_alignment[[
    'event_name', 'event_dau_change_pct',
    'normal_participation_benchmark_index',
    'normal_clear_rate_delta_pp', 'd30_retention_change_pp',
    'alignment_result',
]].round(2)

In [ ]:
save_pve_charts(
    boss_summary, boss_regional, pve_alignment, PROJECT_ROOT / 'images',
)
print('Analysis 3 charts regenerated in:', PROJECT_ROOT / 'images')

## 7. Analysis 4 — revenue growth and subscription value

The launch uses a 30-day local baseline, a 31-day launch window, and a 14-day post-launch guardrail. Revenue per service payer-day uses all daily service PU, not buyers of a specific product, and is not deduplicated period ARPPU. The decomposition is arithmetic rather than causal.

In [ ]:
sales_enriched = prepare_sales(sales, products)
monetization_windows = monetization_window_summary(daily, sales_enriched)
bm_evaluation = bm_evaluation_summary(monetization_windows, retention)
adjacent_products = adjacent_product_summary(daily, sales_enriched)
bm_decomposition = revenue_decomposition(monetization_windows)
bm_evaluation[[
    'scope', 'launch_revenue_per_day_change_pct',
    'launch_pu_per_day_change_pct',
    'adjacent_launch_revenue_per_payer_day_change_pct',
    'adjacent_post_14_revenue_per_payer_day_change_pct',
    'd30_change_pp', 'outcome',
]].round(2)

In [ ]:
save_monetization_charts(
    monetization_windows, bm_evaluation, adjacent_products,
    bm_decomposition, PROJECT_ROOT / 'images',
)
print('Analysis 4 charts regenerated in:', PROJECT_ROOT / 'images')

## 8. Analysis 5 — incident impact and staged recovery

Daily recovery is indexed to the clean July 31–August 11 baseline. Technical availability, activity, paying users, revenue, and monthly D30 are evaluated separately; payer and retention behavior are proxies rather than direct measurements of trust. October overlaps the regional autumn event, so pooled October–November D30 is a cross-context post-recovery check rather than an incident-level causal estimate.

In [ ]:
incident_windows = incident_window_summary(daily)
incident_stages = incident_stage_evaluation(incident_windows)
incident_retention = incident_retention_summary(retention)
incident_final = incident_final_evaluation(incident_stages, incident_retention)
incident_final[[
    'scope', 'dau_recovery_index', 'pu_recovery_index',
    'revenue_recovery_index', 'user_outflow_recovery_index',
    'd30_change_pp', 'final_outcome',
]].round(2)

In [ ]:
save_incident_charts(
    daily, incident_windows, incident_retention, PROJECT_ROOT / 'images',
)
print('Analysis 5 charts regenerated in:', PROJECT_ROOT / 'images')

## 9. Analysis 6 — overall findings and final recommendations

The synthesis preserves each source metric and threshold rather than inventing a composite regional score. Shared guardrail failures drive global actions; unique failures or clearly worst diagnostic gaps drive regional investigation priorities without becoming causal conclusions.

In [ ]:
collaboration_regional = pd.concat([
    regional_quality_summary(retention_monthly, 'fantasy_crossover_2024'),
    regional_quality_summary(retention_monthly, 'astra_crossover_2025'),
], ignore_index=True)
regional_evidence = regional_evidence_summary(
    dependency_results, collaboration_regional, boss_regional,
    bm_evaluation, incident_final,
)
regional_guardrails = regional_guardrail_matrix(regional_evidence)
regional_actions = regional_action_plan(regional_evidence)
regional_actions[['scope', 'priority', 'theme', 'action']]

In [ ]:
save_regional_charts(
    regional_evidence, regional_guardrails, PROJECT_ROOT / 'images',
)
print('Analysis 6 charts regenerated in:', PROJECT_ROOT / 'images')

## 10. Interpretation

- The PvE subscription launch clears the daily-revenue and paying-user thresholds, but the product supplies only 25.5% of the arithmetic lift from the local baseline.
- Adjacent revenue per service payer-day declines 9.9% in the following 14 days, creating a delayed portfolio warning without proving buyer-level switching or an absolute adjacent-revenue collapse.
- The 2025 crossover acquires users but produces weaker D30 retention than the successful 2024 collaboration.
- The service is fully unavailable on August 12, so every observable activity and commerce metric is zero; partial restoration at noon on August 13 completes a 36-hour outage.
- Planned live-ops days concentrate far more revenue than their share of the calendar, while DAU concentration is modest; this is evidence of monetization dependence, not causality.
- Overlapping event windows are explicitly withheld from durability attribution.
- The Fantasy event-context cohort improves D30 quality, while Astra increases cohort volume but fails the D30 guardrail in every region.
- Astra boss entry is below benchmark at every difficulty, while participant clear rate and attempt burden remain comparable; the observed break is pre-entry rather than demonstrated combat difficulty.
- Against the clean pre-incident baseline, compensation raises returned users to 293.9% while revenue reaches only 56.2%, indicating that reactivation outpaces commercial recovery.
- Post-recovery DAU, paying users, and revenue clear their thresholds in every region, while outflow remains above baseline and pooled October–November D30 fails the guardrail everywhere; October overlaps the regional autumn event.
- The postmortem and recurrence-prevention commitment coincides with activity recovery; retention and payer behavior remain only indirect trust proxies.
- Regional execution should rebuild qualified acquisition in KR, protect recurring-product roles in JP, and apply quality gates before further acquisition scale in Global West.

## 11. Scope and limitations

The project excludes gacha pulls, direct limited-character sales, user-level transactions, sentiment logs, and character-level combat records. D30 cohorts are published only through November 2025. Synthetic scenario effects demonstrate analytical design and do not represent industry benchmarks. Formal ARIMA forecasting is excluded because authored interventions and structural breaks dominate the series; rolling trends and event-aware comparisons better match the decision questions.